# Eau et agriculture : qualité des eaux et usage des pesticides

**Problématique :** _à formuler_

Auteurs : _à compléter_

## 0. Imports et configuration

In [ ]:
import os

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src.hubeau.recuperer_toutes_stations import recuperer_toutes_stations
from src.hubeau.recuperer_toutes_analyses import recuperer_toutes_analyses
from src.geo.recuperer_departements import recuperer_departements
from src.nettoyage.nettoyer_analyses import nettoyer_analyses
from src.nettoyage.agreger_eau import agreger_eau
from src.nettoyage.tableau_molecules import tableau_molecules
from src.nettoyage.taux_par_departement import taux_par_departement
from src.visualisation.tracer_taux_quantification import tracer_taux_quantification
from src.visualisation.tracer_carte import tracer_carte
from src.visualisation.tracer_stations import tracer_stations

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e5e5e5",
})

## 1. Récupération des données

1.1 Qualité des cours d'eau (Hub'Eau)  
1.2 Qualité des nappes (Hub'Eau)  
1.3 Ventes de pesticides (BNV-D)  
1.4 Données agricoles (RPG)  
1.5 Fonds de carte

### 1.1 Qualité des cours d'eau (Hub'Eau)

Stations de mesure des 96 départements de métropole.

In [ ]:
CHEMIN_STATIONS = "data/raw/stations_rivieres.parquet"

if os.path.exists(CHEMIN_STATIONS):
    stations = pd.read_parquet(CHEMIN_STATIONS)
else:
    stations = recuperer_toutes_stations()
    stations.to_parquet(CHEMIN_STATIONS, index=False)

stations.shape

In [ ]:
CHEMIN_ANALYSES = "data/raw/analyses_rivieres.parquet"

if os.path.exists(CHEMIN_ANALYSES):
    analyses = pd.read_parquet(CHEMIN_ANALYSES)
else:
    analyses = recuperer_toutes_analyses()
    analyses.to_parquet(CHEMIN_ANALYSES, index=False)

analyses.shape

### 1.5 Fonds de carte

Contours simplifiés des départements ([france-geojson](https://github.com/gregoiredavid/france-geojson)), projetés en Lambert 93 (EPSG:2154), la projection officielle de la métropole.

In [ ]:
CHEMIN_DEPARTEMENTS = "data/geo/departements.parquet"

if os.path.exists(CHEMIN_DEPARTEMENTS):
    departements = gpd.read_parquet(CHEMIN_DEPARTEMENTS)
else:
    departements = recuperer_departements()
    departements.to_parquet(CHEMIN_DEPARTEMENTS)

departements = departements.to_crs("EPSG:2154")

## 2. Nettoyage et mise en forme

### 2.1 Qualité des cours d'eau : `eau.parquet`

On garde les résultats en µg/L, on retire les analyses non faites et les doublons, puis on agrège au département × année × molécule (voir `docs/contrat_donnees.md`).

In [ ]:
analyses_propres = nettoyer_analyses(analyses)
eau = agreger_eau(analyses_propres)
eau.to_parquet("data/processed/eau.parquet", index=False)

print("doublons sur la clé :", eau.duplicated(subset=["code_dep", "annee", "milieu", "code_sandre"]).sum())
eau.shape

## 3. Statistiques descriptives

On ajoute le nom des molécules (`molecules.csv`) et le mois de prélèvement. `eau.parquet` est déjà enregistré : ces colonnes ne servent qu'à l'analyse.

In [ ]:
molecules = pd.read_csv("data/reference/molecules.csv", dtype={"code_sandre": str})
noms = molecules[["code_sandre", "molecule", "molecule_mere"]]

eau = eau.merge(noms)
analyses_propres = analyses_propres.merge(noms)
analyses_propres["mois"] = analyses_propres["date_prelevement"].str[5:7].astype(int)

### 3.1 Couverture des données

Nombre de stations suivies et d'analyses par année, toutes molécules confondues.

In [ ]:
analyses_propres.groupby("annee").agg(
    nb_stations=("code_station", "nunique"),
    nb_analyses=("code_station", "size"),
)

Environ 4 000 à 4 700 stations et 200 000 analyses par an. 2015 est moins couverte (3 562 stations) : certaines molécules, comme l'ESA-métolachlore, ne sont alors suivies que dans 51 départements. Les comparaisons entre années se font donc en **taux**, pas en nombres d'analyses.

### 3.2 Indicateurs par molécule (2015-2022)

`part_depassements` : part des analyses au-dessus de 0,1 µg/L, la norme de l'eau potable pour un pesticide.

In [ ]:
tableau_molecules(eau).round(3)

- Les **métabolites** dominent : l'AMPA (glyphosate) et l'ESA-métolachlore sont quantifiés dans environ deux analyses sur trois et dépassent 0,1 µg/L dans 4 cas sur 10. Ils persistent plus longtemps que leur molécule mère. L'AMPA vient aussi en partie des lessives.
- L'**atrazine** et la déséthylatrazine, interdites depuis 2003, restent souvent quantifiées mais presque jamais au-dessus de 0,1 µg/L : c'est une pollution d'héritage.
- Les **maximums** sont très sensibles aux valeurs aberrantes (1 541 µg/L de glyphosate dans la Manche en 2019) : on s'appuie plutôt sur le taux de quantification et la part de dépassements.

### 3.3 Évolution 2015-2022

Taux de quantification par année. En bleu la molécule vendue, en orange son produit de dégradation (métabolite).

In [ ]:
tracer_taux_quantification(analyses_propres, "annee");

Pas de tendance nette sur 2015-2022 : les taux restent stables. Le bond de l'ESA-métolachlore entre 2015 et 2016 vient surtout de l'extension de sa surveillance, pas d'une hausse de la pollution. L'atrazine baisse lentement après 2017.

### 3.4 Saisonnalité

Taux de quantification par mois de prélèvement, toutes années confondues.

In [ ]:
tracer_taux_quantification(analyses_propres, "mois");

La saisonnalité suit le calendrier des épandages :
- **S-métolachlore** (maïs, semis au printemps) : pic en **mai-juin** ;
- **prosulfocarbe et chlortoluron** (céréales d'hiver, semis à l'automne) : pic d'**octobre à janvier**, quasi absents l'été ;
- **atrazine** : aucune saisonnalité, puisqu'elle n'est plus épandue.

Les rivières réagissent donc dans les semaines qui suivent l'usage : c'est ce qui justifie de croiser la qualité de l'eau avec les ventes **de la même année**.

## 4. Visualisations et cartes

### 4.1 Stations suivies

Stations ayant au moins une analyse d'une des 8 molécules entre 2015 et 2022.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
tracer_stations(stations[stations["code_station"].isin(analyses_propres["code_station"])], departements, ax)
ax.set_title("Stations suivies (2015-2022)");

Le réseau couvre tout le territoire, avec une densité plus forte en Bretagne, dans le Nord et dans le Bassin parisien.

### 4.2 Effort de surveillance

Nombre d'analyses par département, toutes molécules et années confondues.

In [ ]:
analyses_par_dep = eau.groupby("code_dep", as_index=False)["nb_analyses"].sum()

fig, ax = plt.subplots(figsize=(6, 6))
tracer_carte(departements, analyses_par_dep, "nb_analyses", "Nombre d'analyses (2015-2022)", ax)

L'effort de surveillance est très inégal : la Bretagne (Côtes-d'Armor, Ille-et-Vilaine) compte plus de 35 000 analyses, contre quelques milliers dans les Alpes ou en Corse. Un département très surveillé détecte mécaniquement plus de molécules : c'est une raison de plus pour raisonner en taux.

### 4.3 Où trouve-t-on quelles molécules ?

Taux de quantification par département (2015-2022) pour un herbicide du maïs (via son métabolite) et un herbicide des céréales d'hiver. Chaque carte a sa propre échelle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
tracer_carte(departements, taux_par_departement(eau, "ESA-metolachlore"), "taux_quantification",
             "ESA-métolachlore (maïs)", axes[0])
tracer_carte(departements, taux_par_departement(eau, "prosulfocarbe"), "taux_quantification",
             "Prosulfocarbe (céréales d'hiver)", axes[1])

Les deux cartes dessinent des zones différentes, qui correspondent aux cultures :
- l'**ESA-métolachlore** (maïs) est surtout quantifié dans l'Ouest (Bretagne, Pays de la Loire) et le Sud-Ouest (Landes) ;
- le **prosulfocarbe** (céréales d'hiver) se concentre dans le Nord et le Bassin parisien.

Le croisement avec les surfaces cultivées (RPG) et les ventes (BNV-D) devra confirmer ce lien.

## 5. Croisement eau × agriculture

## 6. Conclusion et limites